In [ ]:
import sys; sys.path.append('..')
import mesh, elastic_sheet, energy, benchmark
import triangulation
from tri_mesh_viewer import TriMeshViewer
import numpy as np

In [ ]:
V, E = mesh.load_raw('Data/victorinox.obj')
#V, F, sm = triangulation.triangulate(V[:, 0:2], E, triArea=0.25)
V, F, sm = triangulation.triangulate(V[:, 0:2], E, triArea=1.0)
sm = np.array(sm)
m = mesh.Mesh(V, F)

In [ ]:
isBoundary = np.zeros(m.numVertices(), dtype=np.bool)
isBoundary[m.boundaryVertices()] = True

In [ ]:
creases = []
def identifyCreases(edge, eidx):
    edge = list(edge)
    if sm[edge].all() and not isBoundary[edge].all():
        creases.append(edge)
m.visitEdges(identifyCreases)
creases = np.array(creases, dtype=np.int)

In [ ]:
psi = energy.NeoHookeanYoungPoisson(2, 1, 0.0)
es = elastic_sheet.ElasticSheet(m, psi, creases)
es.thickness = 0.05

In [ ]:
esview = TriMeshViewer(es, wireframe=True, width=1024, height=768)

In [ ]:
esview.materialLibrary.material(False).color='#CC1111'

In [ ]:
esview.show()

In [ ]:
pinVars, pinVerts = es.prepareRigidMotionPins()

In [ ]:
creaseVars = np.arange(es.numCreases()) + es.creaseAngleOffset()

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.niter = 20

In [ ]:
esview.update()

In [ ]:
es.setCreaseAngles(es.getCreaseAngles()[0] - 0.05 * np.pi / 16 * np.ones(es.numCreases()))
def iter_cb(prob, it):
    pass
    #if (it % 4 == 1):
        #esview.update()
benchmark.reset()
es.computeEquilibrium(loads=[], fixedVars=pinVars + list(creaseVars), cb=iter_cb, opts=opts)
esview.update()
benchmark.report()

In [ ]:
esview.update()